# Project 3 — Early Warning System for Sepsis (PhysioNet 2019)
## Notebook 03 — Grouped CV Modeling + OOF Predictions (for Calibration & Policy)

**Goal**
Train models using **patient-grouped** cross-validation and generate **out-of-fold (OOF)** probability predictions.
These OOF predictions are the only correct input for Notebook 04 (calibration + alert policy).

**Input**
- `results/features_02.parquet`

**Outputs**
- `results/oof_predictions.parquet` (patient_id, ICULOS, y, p_oof, fold_id, model_name)
- `results/model_metrics_cv.json` (AUROC/AUPRC per fold + mean/std)
- (optional) `results/final_model.pkl` (trained on all data)

**Key rule**
All splits must be grouped by `patient_id`.


In [19]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

pd.set_option("display.max_columns", 200)

# -------------------------
# Paths (EDIT THIS ONCE)
# -------------------------
PROJECT_ROOT = Path(".").resolve()
DATA_ROOT = PROJECT_ROOT  # set this if your results live elsewhere

RESULTS_DIR = DATA_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IN_FEATS = RESULTS_DIR / "features_02b.parquet"
OUT_OOF = RESULTS_DIR / "oof_predictions.parquet"
OUT_METRICS = RESULTS_DIR / "model_metrics_cv.json"

print("Input:", IN_FEATS, "exists:", IN_FEATS.exists())


Input: C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results\features_02b.parquet exists: True


### 1) Load engineered features
We keep:
- `patient_id`, `ICULOS` as identifiers (needed later for policy & lead-time)
- `y_within_H` as the target


In [20]:
df = pd.read_parquet(IN_FEATS)
df = df.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)

required = {"patient_id", "ICULOS", "y_within_H"}
missing_req = required - set(df.columns)
assert not missing_req, f"Missing required columns: {missing_req}"

print("Rows:", len(df), "Patients:", df["patient_id"].nunique(), "Cols:", df.shape[1])
print("Target prevalence (row-level):", float(df["y_within_H"].mean()))
df.head()


Rows: 19145 Patients: 493 Cols: 675
Target prevalence (row-level): 0.011177853225385219


,patient_id,ICULOS,y_within_H,HR__miss,O2Sat__miss,Temp__miss,SBP__miss,MAP__miss,DBP__miss,Resp__miss,EtCO2__miss,BaseExcess__miss,HCO3__miss,FiO2__miss,pH__miss,PaCO2__miss,SaO2__miss,AST__miss,BUN__miss,Alkalinephos__miss,Calcium__miss,Chloride__miss,Creatinine__miss,Bilirubin_direct__miss,Glucose__miss,Lactate__miss,Magnesium__miss,Phosphate__miss,Potassium__miss,Bilirubin_total__miss,TroponinI__miss,Hct__miss,Hgb__miss,PTT__miss,WBC__miss,Fibrinogen__miss,Platelets__miss,Age__miss,Gender__miss,Unit1__miss,Unit2__miss,HospAdmTime__miss,SepsisLabel__miss,event_iculos__miss,use_row__miss,HR__mean_6,O2Sat__mean_6,Temp__mean_6,SBP__mean_6,MAP__mean_6,DBP__mean_6,Resp__mean_6,EtCO2__mean_6,BaseExcess__mean_6,HCO3__mean_6,FiO2__mean_6,pH__mean_6,PaCO2__mean_6,SaO2__mean_6,AST__mean_6,BUN__mean_6,Alkalinephos__mean_6,Calcium__mean_6,Chloride__mean_6,Creatinine__mean_6,Bilirubin_direct__mean_6,Glucose__mean_6,Lactate__mean_6,Magnesium__mean_6,Phosphate__mean_6,Potassium__mean_6,Bilirubin_total__mean_6,TroponinI__mean_6,Hct__mean_6,Hgb__mean_6,PTT__mean_6,WBC__mean_6,Fibrinogen__mean_6,Platelets__mean_6,Age__mean_6,Gender__mean_6,Unit1__mean_6,Unit2__mean_6,HospAdmTime__mean_6,SepsisLabel__mean_6,event_iculos__mean_6,use_row__mean_6,HR__min_6,O2Sat__min_6,Temp__min_6,SBP__min_6,MAP__min_6,DBP__min_6,Resp__min_6,EtCO2__min_6,BaseExcess__min_6,HCO3__min_6,FiO2__min_6,pH__min_6,PaCO2__min_6,...,Bilirubin_total__delta_1h,TroponinI__delta_1h,Hct__delta_1h,Hgb__delta_1h,PTT__delta_1h,WBC__delta_1h,Fibrinogen__delta_1h,Platelets__delta_1h,Age__delta_1h,Gender__delta_1h,Unit1__delta_1h,Unit2__delta_1h,HospAdmTime__delta_1h,SepsisLabel__delta_1h,event_iculos__delta_1h,use_row__delta_1h,HR__delta_6h,O2Sat__delta_6h,Temp__delta_6h,SBP__delta_6h,MAP__delta_6h,DBP__delta_6h,Resp__delta_6h,EtCO2__delta_6h,BaseExcess__delta_6h,HCO3__delta_6h,FiO2__delta_6h,pH__delta_6h,PaCO2__delta_6h,SaO2__delta_6h,AST__delta_6h,BUN__delta_6h,Alkalinephos__delta_6h,Calcium__delta_6h,Chloride__delta_6h,Creatinine__delta_6h,Bilirubin_direct__delta_6h,Glucose__delta_6h,Lactate__delta_6h,Magnesium__delta_6h,Phosphate__delta_6h,Potassium__delta_6h,Bilirubin_total__delta_6h,TroponinI__delta_6h,Hct__delta_6h,Hgb__delta_6h,PTT__delta_6h,WBC__delta_6h,Fibrinogen__delta_6h,Platelets__delta_6h,Age__delta_6h,Gender__delta_6h,Unit1__delta_6h,Unit2__delta_6h,HospAdmTime__delta_6h,SepsisLabel__delta_6h,event_iculos__delta_6h,use_row__delta_6h,HR__prev1,O2Sat__prev1,Temp__prev1,SBP__prev1,MAP__prev1,DBP__prev1,Resp__prev1,EtCO2__prev1,BaseExcess__prev1,HCO3__prev1,FiO2__prev1,pH__prev1,PaCO2__prev1,SaO2__prev1,AST__prev1,BUN__prev1,Alkalinephos__prev1,Calcium__prev1,Chloride__prev1,Creatinine__prev1,Bilirubin_direct__prev1,Glucose__prev1,Lactate__prev1,Magnesium__prev1,Phosphate__prev1,Potassium__prev1,Bilirubin_total__prev1,TroponinI__prev1,Hct__prev1,Hgb__prev1,PTT__prev1,WBC__prev1,Fibrinogen__prev1,Platelets__prev1,Age__prev1,Gender__prev1,Unit1__prev1,Unit2__prev1,HospAdmTime__prev1,SepsisLabel__prev1,event_iculos__prev1,use_row__prev1
0,p000214,2,0,0,0,1,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,0,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1.0,NaN,NaN,-13.07,0.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.53,1.0,NaN,NaN,-13.07,0.0,NaN,True
1,p000214,3,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,0,0,1,1,0,0,1,0,76.000000,100.0,NaN,143.0,91.000000,65.000000,16.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

### 2) Define X / y / groups
We exclude identifiers and target from X.


In [21]:
ID_COLS = ["patient_id", "ICULOS"]
TARGET_COL = "y_within_H"

feature_cols = [c for c in df.columns if c not in ID_COLS + [TARGET_COL]]
X = df[feature_cols]
y = df[TARGET_COL].astype(int).values
groups = df["patient_id"].values

print("n_features:", len(feature_cols))
print("Example features:", feature_cols[:10])


n_features: 672
Example features: ['HR__miss', 'O2Sat__miss', 'Temp__miss', 'SBP__miss', 'MAP__miss', 'DBP__miss', 'Resp__miss', 'EtCO2__miss', 'BaseExcess__miss', 'HCO3__miss']


### 3) Helpers: grouped CV + OOF predictions
We compute AUROC & AUPRC per fold and save OOF probabilities.


In [22]:
def make_sample_weight(y_fold: np.ndarray) -> np.ndarray:
    # Simple inverse-frequency weights (helps imbalance)
    n_pos = (y_fold == 1).sum()
    n_neg = (y_fold == 0).sum()
    if n_pos == 0:
        return np.ones_like(y_fold, dtype=float)
    w_pos = n_neg / max(n_pos, 1)
    w = np.ones_like(y_fold, dtype=float)
    w[y_fold == 1] = w_pos
    return w

def run_group_cv_oof(model, X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, n_splits=5, model_name="model"):
    """
    Patient-grouped CV with true out-of-fold (OOF) probabilities.

    Notes:
    - Uses sklearn.clone() so each fold gets a fresh estimator.
    - Passes sample weights robustly:
        * for Pipelines with final step named 'clf', uses clf__sample_weight
        * otherwise uses sample_weight if supported
    """
    gkf = GroupKFold(n_splits=n_splits)

    oof = np.full(shape=len(y), fill_value=np.nan, dtype=float)
    fold_ids = np.full(shape=len(y), fill_value=-1, dtype=int)

    fold_metrics = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        sw = make_sample_weight(y_tr)

        m = clone(model)

        # Fit (try best-effort weighted fit)
        fit_ok = False
        try:
            m.fit(X_tr, y_tr, clf__sample_weight=sw)  # Pipeline case (last step 'clf')
            fit_ok = True
        except TypeError:
            pass
        if not fit_ok:
            try:
                m.fit(X_tr, y_tr, sample_weight=sw)     # Estimator supports sample_weight directly
                fit_ok = True
            except TypeError:
                m.fit(X_tr, y_tr)                       # Unweighted fallback

        # Predict probability for positive class
        if hasattr(m, "predict_proba"):
            p = m.predict_proba(X_va)[:, 1]
        else:
            s = m.decision_function(X_va)
            p = 1 / (1 + np.exp(-s))

        oof[va_idx] = p
        fold_ids[va_idx] = fold

        auroc = roc_auc_score(y_va, p) if len(np.unique(y_va)) > 1 else np.nan
        auprc = average_precision_score(y_va, p) if len(np.unique(y_va)) > 1 else np.nan

        fold_metrics.append({
            "model": model_name,
            "fold": int(fold),
            "auroc": float(auroc),
            "auprc": float(auprc),
            "n_val": int(len(va_idx)),
            "pos_val": int(y_va.sum())
        })

        print(f"[{model_name}] Fold {fold}: AUROC={auroc:.4f} AUPRC={auprc:.4f}  (val n={len(va_idx)} pos={y_va.sum()})")

    assert np.isfinite(oof).all(), "OOF contains NaNs — check splits/training."
    return oof, fold_ids, fold_metrics


### 4) Models
We include:
- **Logistic Regression** baseline (interpretable)
- **HistGradientBoosting** (strong non-linear baseline, fast, no GPU)

You can swap HGB later for LightGBM/CatBoost if you decide to push peak performance.


In [23]:
RANDOM_SEED = 42
N_SPLITS = 5

# Logistic Regression pipeline
lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),  # safe for sparse-ish wide data
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs"
    )),
])

# HistGradientBoosting
hgb = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=400,
        min_samples_leaf=50,
        l2_regularization=0.0,
        random_state=RANDOM_SEED
    )),
])

models = [
    ("logreg", lr),
    ("hgb", hgb),
]


### 5) Run grouped CV + produce OOF predictions
We will save OOF predictions for **both** models, then choose the best for Notebook 04.


In [24]:
all_metrics = []
oof_frames = []

for name, model in models:
    oof, fold_ids, metrics = run_group_cv_oof(model, X, y, groups, n_splits=N_SPLITS, model_name=name)
    all_metrics.extend(metrics)

    oof_df = df[["patient_id", "ICULOS"]].copy()
    oof_df["y"] = y
    oof_df["p_oof"] = oof
    oof_df["fold_id"] = fold_ids
    oof_df["model_name"] = name
    oof_frames.append(oof_df)

metrics_df = pd.DataFrame(all_metrics)
metrics_df


[logreg] Fold 0: AUROC=0.9168 AUPRC=0.0576  (val n=3830 pos=24)
[logreg] Fold 1: AUROC=0.9515 AUPRC=0.1793  (val n=3829 pos=38)
[logreg] Fold 2: AUROC=0.9492 AUPRC=0.2319  (val n=3829 pos=48)
[logreg] Fold 3: AUROC=0.7830 AUPRC=0.1838  (val n=3829 pos=42)
[logreg] Fold 4: AUROC=0.8859 AUPRC=0.1390  (val n=3828 pos=62)
[hgb] Fold 0: AUROC=0.9628 AUPRC=0.2264  (val n=3830 pos=24)
[hgb] Fold 1: AUROC=0.9757 AUPRC=0.2839  (val n=3829 pos=38)
[hgb] Fold 2: AUROC=0.9140 AUPRC=0.0679  (val n=3829 pos=48)
[hgb] Fold 3: AUROC=0.9328 AUPRC=0.1908  (val n=3829 pos=42)
[hgb] Fold 4: AUROC=0.9172 AUPRC=0.1242  (val n=3828 pos=62)


,model,fold,auroc,auprc,n_val,pos_val
0,logreg,0,0.916798,0.057570,3830,24
1,logreg,1,0.951547,0.179347,3829,38
2,logreg,2,0.949231,0.231911,3829,48
3,logreg,3,0.782973,0.183771,3829,42
4,logreg,4,0.885949,0.138975,3828,62
5,hgb,0,0.962811,0.226377,3830,24
6,hgb,1,0.975656,0.283879,3829,38
7,hgb,2,0.914011,0.067948,3829,48
8,hgb,3,0.932847,0.190810,3829,42
9,hgb,4,0.917188,0.124174,3828,62


In [25]:
# Summarize performance
summary = (metrics_df.groupby("model")[["auroc", "auprc"]]
           .agg(["mean", "std"])
          )
summary


auroc               auprc          
            mean       std      mean       std
model                                         
hgb     0.940502  0.027556  0.178638  0.084737
logreg  0.897300  0.069317  0.158315  0.065250

### 6) Save artifacts
Notebook 04 will load `oof_predictions.parquet` and work only with these OOF probabilities.


In [26]:
oof_all = pd.concat(oof_frames, ignore_index=True)
oof_all.to_parquet(OUT_OOF, index=False)

# metrics json
out = {
    "n_rows": int(len(df)),
    "n_patients": int(df["patient_id"].nunique()),
    "target_prevalence_row": float(df["y_within_H"].mean()),
    "n_splits": int(N_SPLITS),
    "metrics_per_fold": all_metrics,
    "summary": {
        m: {
            "auroc_mean": float(metrics_df.loc[metrics_df["model"]==m, "auroc"].mean()),
            "auroc_std":  float(metrics_df.loc[metrics_df["model"]==m, "auroc"].std()),
            "auprc_mean": float(metrics_df.loc[metrics_df["model"]==m, "auprc"].mean()),
            "auprc_std":  float(metrics_df.loc[metrics_df["model"]==m, "auprc"].std()),
        }
        for m in metrics_df["model"].unique()
    }
}
OUT_METRICS.write_text(json.dumps(out, indent=2))

print("Saved:")
print(" -", OUT_OOF)
print(" -", OUT_METRICS)


Saved:
 - C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results\oof_predictions.parquet
 - C:\Users\Admin\anaconda_projects\f41dcae4-d989-44e3-8a04-984a86780085\notebooks\results\model_metrics_cv.json


### 7) Choose the model for Notebook 04
Set `BEST_MODEL_NAME` to the model you will calibrate + use in the alert policy.

Tip: Often the best choice is the model with the best **AUPRC** (since positives are rare).


In [27]:
BEST_MODEL_NAME = "hgb"  # change to "logreg" if it wins on your data

best_oof = oof_all[oof_all["model_name"] == BEST_MODEL_NAME].copy()
print(best_oof.head())
print("Best model rows:", len(best_oof))
print("Best model OOF AUROC:", roc_auc_score(best_oof["y"], best_oof["p_oof"]))
print("Best model OOF AUPRC:", average_precision_score(best_oof["y"], best_oof["p_oof"]))


      patient_id  ICULOS  y     p_oof  fold_id model_name
19145    p000214       2  0  0.000250        4        hgb
19146    p000214       3  0  0.000241        4        hgb
19147    p000214       4  0  0.000262        4        hgb
19148    p000214       5  0  0.000262        4        hgb
19149    p000214       6  0  0.000262        4        hgb
Best model rows: 19145
Best model OOF AUROC: 0.9336895869258601
Best model OOF AUPRC: 0.1288055734449206
